# 01 · Instruction Fine-Tuning
### *Aligning & Deploying LLMs — Unit 2*

A base language model is trained to **continue text**. Ask it a question and it often just continues the
*pattern* instead of answering. **Instruction fine-tuning (SFT)** continues training the model on
`(instruction → ideal response)` pairs so it learns the *behaviour* of following instructions.

In this notebook you will:
1. Watch a raw base model (`distilgpt2`) **continue** a prompt instead of answering it.
2. Format data in the **Alpaca instruction template**.
3. Run a small **supervised fine-tune**.
4. **Compare** the model before and after.

> **Runtime:** `Runtime → Change runtime type → GPU` (T4) makes training much faster, but this will also run on CPU.

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.19" accelerate

## 1 · How a *base* model behaves

`distilgpt2` is a small **base** model — pretrained only to predict the next token. Notice how it does **not**
answer the question; it drifts into more text that merely looks like a plausible continuation.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "distilgpt2"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL)

def generate(model, prompt, max_new_tokens=45):
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)

print("PROMPT: List three uses of solar power.\n")
print("BASE MODEL SAYS:")
print(generate(base, "List three uses of solar power."))

## 2 · The instruction template

Instruction tuning reshapes every task into the same skeleton. We use the well-known **Alpaca** template:
an `### Instruction:` block, an optional `### Input:` block, and a `### Response:` block the model must learn to produce.

In [ ]:
PROMPT_WITH_INPUT = (
    "Below is an instruction that describes a task, paired with an input. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n"
)
PROMPT_NO_INPUT = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n"
)

def build_prompt(instruction, inp=""):
    if inp and inp.strip():
        return PROMPT_WITH_INPUT.format(instruction=instruction, input=inp)
    return PROMPT_NO_INPUT.format(instruction=instruction)

print(build_prompt("Translate to French.", "Good morning") + "Bonjour")

## 3 · A small instruction dataset

We take a **small slice** of the [Alpaca](https://huggingface.co/datasets/tatsu-lab/alpaca) dataset
(52k instruction/response pairs) so it fine-tunes quickly. Each example becomes
`prompt + response + <eos>`, and the labels equal the input ids (standard causal-LM SFT).

In [ ]:
from datasets import load_dataset

raw = load_dataset("tatsu-lab/alpaca", split="train").shuffle(seed=0).select(range(800))

def format_example(ex):
    text = build_prompt(ex["instruction"], ex["input"]) + ex["output"] + tok.eos_token
    return {"text": text}

ds = raw.map(format_example, remove_columns=raw.column_names)

def tokenize(ex):
    out = tok(ex["text"], truncation=True, max_length=256, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

tokenized = ds.map(tokenize, remove_columns=["text"])
print(tokenized)
print("\nExample text:\n", ds[0]["text"][:400])

## 4 · Supervised fine-tune (SFT)

A short training run — a few hundred steps is enough to *see the behaviour change* on a tiny model.
This is ordinary supervised learning; the only trick is the **instruction format** of the data.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

model = AutoModelForCausalLM.from_pretrained(MODEL)
collator = DataCollatorForLanguageModeling(tok, mlm=False)

args = TrainingArguments(
    output_dir="sft-distilgpt2",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    report_to="none",
    save_strategy="no",
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
trainer.train()

## 5 · Before vs after

Same prompt, same decoding — but the fine-tuned model now treats the line as a **request to fulfil**
rather than a pattern to continue. (On a 82M-param model the answers are modest; the *shift in behaviour*
is the point.)

In [ ]:
tuned = trainer.model.eval()

for instr in ["List three uses of solar power.",
              "Give one tip for writing clearly.",
              "Name a fruit that is red."]:
    p = build_prompt(instr)
    print("="*70)
    print("INSTRUCTION:", instr)
    print("BASE :", generate(base, instr).replace("\n", " ")[:160])
    print("TUNED:", generate(tuned, p).replace("\n", " ")[:160])

## Recap & your turn

- Pretraining teaches **language**; instruction tuning teaches the model to **do the task you asked**.
- The data does the work — the training loop is plain supervised learning.
- **Task diversity** (many kinds of instruction), not just volume, is what drives generalization to *unseen* instructions.

**Exercises**
1. Increase the slice to `range(4000)` and epochs to 3 — do responses improve?
2. Swap the base for `EleutherAI/pythia-160m` and compare.
3. Add your own 10 hand-written examples in the Alpaca format and mix them in.
4. Try **LoRA** (parameter-efficient tuning) with `peft` to fine-tune a larger model on free Colab.